# OLMo Lineage Fingerprint Verification

Read a lineage verification JSON report, normalize each fingerprint technique into tidy summary tables, and plot replay success or LLMmap closeness across the configured lineage order.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

REPO_ROOT = Path("..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.verification.plot_lineage_verification import (
    load_report,
    normalize_report,
    plot_llmmap_reference_closeness,
    plot_llmmap_reference_distances,
    plot_replay_match_rates,
)

# Reuse this notebook with another newer lineage report by changing this path.
REPORT_PATH = Path("../artifacts/verification/olmo2_instruct_reference_trajectory.json")

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("default")

## Load and Normalize

This notebook intentionally expects the newer lineage report shape from `scripts/verification/verify_fingerprint_lineage.py`: top-level `targets`, `target_metadata`, and per-technique results keyed by target keys such as `instruct@step_200`.

In [ ]:
# Normalization and plotting helpers are imported from scripts.verification.plot_lineage_verification.


In [ ]:
try:
    report = load_report(REPORT_PATH)
    lineage_df, replay_df, llmmap_df = normalize_report(report)
    print(f"Loaded {REPORT_PATH}")
    print(f"Targets: {len(lineage_df)}")
    print(f"Replay rows: {len(replay_df)}")
    print(f"LLMmap rows: {len(llmmap_df)}")
except FileNotFoundError as exc:
    report = None
    lineage_df = pd.DataFrame()
    replay_df = pd.DataFrame()
    llmmap_df = pd.DataFrame()
    print(exc)
    print("Set REPORT_PATH to a newer lineage verification report, then rerun from this cell.")

## Replay Fingerprints

For summary-style replay fingerprints such as ProFLingo and TRAP, `match_rate` is the reported verification success rate. ProFLingo rows come from delegated `copyright_test.py` keyword-ASR summaries and do not include question-by-question details.

In [ ]:
plot_replay_match_rates(replay_df, lineage_df);

## LLMmap Reference Closeness

LLMmap reports a nearest-template list. The plot uses the reference model's rank in `top_k`; if the reference is absent, it is plotted at `top_k + 1` and labeled as outside the returned candidates. Points marked with `x` are revisions where `matched_reference_top1` is false.

In [ ]:
plot_llmmap_reference_closeness(llmmap_df, lineage_df)
plot_llmmap_reference_distances(llmmap_df, lineage_df)

## Summary Tables

In [ ]:
if not replay_df.empty:
    display(
        replay_df.sort_values(["technique", "order"])[
            [
                "technique",
                "target_key",
                "model_id",
                "revision",
                "step",
                "matched",
                "total",
                "match_rate",
                "verification_mode",
            ]
        ]
    )

if not llmmap_df.empty:
    display(
        llmmap_df.sort_values("order")[
            [
                "target_key",
                "model_id",
                "revision",
                "step",
                "matched_reference_top1",
                "top1_label",
                "top1_distance",
                "reference_rank_in_top_k",
                "reference_rank_label",
                "reference_distance",
            ]
        ]
    )